# 03 — Modelagem

**Dimensão 5 da rúbrica — 20 pontos.**

Mínimo de **dois** classificadores distintos. Um único modelo zera 8 dos 20 pontos.

In [23]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

#Leitura dos arquivos processados no notebook 1 e salvos no Github
url_base_credito = f"https://github.com/camilavbr/Tech_Challenge_Fase2_rm377839/raw/main/data/processed/base_credito.csv"
url_base_credito_faixas = f"https://github.com/camilavbr/Tech_Challenge_Fase2_rm377839/raw/main/data/processed/base_credito_faixas.csv"

df = pd.read_csv(url_base_credito)
df_faixas = pd.read_csv(url_base_credito_faixas)


## 1. Split treino/teste

`stratify` preserva a proporção das classes nos dois conjuntos.

In [24]:
#Segmentação dos dados com 80% para treino e 20% para teste utilizando o stratify para manter as proporções do target no treino e no teste
from sklearn.model_selection import train_test_split

TARGET = "mau_pagador"

x = df.drop(columns=[TARGET])
y = df[TARGET]
x_train, x_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

In [25]:
#Segmentação dos dados por faixa
from sklearn.model_selection import train_test_split

TARGET = "mau_pagador"

x_faixas = df_faixas.drop(columns=[TARGET])
y_faixas = df_faixas[TARGET]
x_faixas_train, x_faixas_test, y_faixas_train, y_faixas_test = train_test_split(
x_faixas, y_faixas, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

## 2. Modelos candidatos

Usar `Pipeline` evita vazamento: o scaler é ajustado só no fold de treino durante a validação cruzada.

In [26]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

modelos = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "random_forest": Pipeline([
        ("scaler", StandardScaler()),
       ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
    ])
}

## 3. Validação cruzada

In [33]:
from sklearn.model_selection import cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados = []

cenarios = {
    'Base Original': (x_train, y_train),
    'Base de Faixas': (x_faixas_train, y_faixas_train)
}

for nome_base, (x_tr, y_tr) in cenarios.items():
    for nome_modelo, modelo in modelos.items():

        # Regressão Logística
        # padronização dos dados através do StandardScaler
        if nome_modelo == 'logistic_regression' and 'Original' in nome_base:
            scaler = StandardScaler()
            x_input = scaler.fit_transform(x_tr)
        else:
            x_input = x_tr

        scores = cross_validate(modelo, x_input, y_tr, cv=cv, scoring=['roc_auc', 'recall', 'f1'], n_jobs=-1)

        resultados.append({
            'Abordagem de Dados': nome_base,
            'Algoritmo': nome_modelo,
            'AUC-ROC Médio': np.mean(scores['test_roc_auc']),
            'Recall Médio (Classe 1)': np.mean(scores['test_recall']),
            'F1-Score Médio': np.mean(scores['test_f1'])
        })

## 4. Comparação


In [34]:
df_comparativo = pd.DataFrame(resultados)
print("📊 RESULTADO DO EXPERIMENTO SOB VALIDAÇÃO CRUZADA:")
display(df_comparativo.sort_values(by='AUC-ROC Médio', ascending=False))

📊 RESULTADO DO EXPERIMENTO SOB VALIDAÇÃO CRUZADA:


,Abordagem de Dados,Algoritmo,AUC-ROC Médio,Recall Médio (Classe 1),F1-Score Médio
1,Base Original,random_forest,0.794520,0.243249,0.275375
3,Base de Faixas,random_forest,0.779146,0.239208,0.269955
0,Base Original,logistic_regression,0.584289,0.000000,0.000000
2,Base de Faixas,logistic_regression,0.577367,0.000000,0.000000


Resultado: O algoritmo Random Forest aplicado à Base Original foi o vencedor, atingindo um AUC-ROC Médio de 0.794 e um Recall de 24,32% sob validação cruzada de 5 dobras.

A Regressão Logística falhou diante do cenário desbalanceado, apresentando Recall de 0.0% por classificar toda a base como classe majoritária (0). O Random Forest mostrou-se superior ao lidar com as relações não-lineares dos dados socioeconômicos e extrair padrões finos das variáveis contínuas brutas, superando a descontinuação por faixas.
